In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import random

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ======================================================
# 1. LOAD DATA
# ======================================================
data_path_x = '/content/drive/MyDrive/HMFEADataset/Full_X_aug.npy'
data_path_y = '/content/drive/MyDrive/HMFEADataset/Full_Y_aug.npy'

x_data = np.load(data_path_x)
y_data = np.load(data_path_y)

print("Original x_data shape:", x_data.shape)
print("y_data shape:", y_data.shape)

x_data = np.squeeze(x_data, axis=-1).astype(np.float32) / 255.0

num_subjects, num_slices, H, W = x_data.shape
x_flat = x_data.reshape(num_subjects * num_slices, H, W)
y_flat = np.repeat(y_data, num_slices)

print(f"Flattened: {x_flat.shape}, labels: {y_flat.shape}")

indices = np.arange(x_flat.shape[0])
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42, stratify=y_flat)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=y_flat[temp_idx])

classes = np.unique(y_flat)
class_weights = compute_class_weight('balanced', classes=classes, y=y_flat[train_idx])
class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights)}
pos_weight = torch.tensor(class_weight_dict[1] / class_weight_dict[0] if class_weight_dict[0] != 0 else 10.0).to(device)

print("Class weights:", class_weight_dict)
print("Pos weight:", pos_weight.item())

# ======================================================
# 2. Dataset & DataLoader
# ======================================================
class SliceDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img = x_flat[self.indices[idx]]
        label = int(y_flat[self.indices[idx]])
        img = self.transform(img)
        return img, label

batch_size = 4
train_loader = DataLoader(SliceDataset(train_idx), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(SliceDataset(val_idx),   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SliceDataset(test_idx),  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

# ======================================================
# 3. DropBlock2D
# ======================================================
class DropBlock2D(nn.Module):
    def __init__(self, block_size=7, keep_prob=0.9):
        super().__init__()
        self.block_size = block_size
        self.keep_prob = keep_prob

    def forward(self, x):
        if not self.training or self.keep_prob == 1.0:
            return x
        b, c, h, w = x.shape
        gamma = (1 - self.keep_prob) * (h * w) / (self.block_size**2) / ((h - self.block_size + 1) * (w - self.block_size + 1))
        mask = torch.bernoulli(torch.full((b, c, h, w), gamma, device=x.device))
        mask = 1 - F.max_pool2d(mask, kernel_size=self.block_size, stride=1, padding=self.block_size//2)
        mask = mask / (mask.mean([1,2,3], keepdim=True) + 1e-8)
        return x * mask

# ======================================================
# 4. Residual Block
# ======================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1, drop_prob=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.drop = DropBlock2D(block_size=7, keep_prob=1-drop_prob)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.drop(out)
        out += identity
        return self.relu(out)

# ======================================================
# 5. DBCNN Branch & Stacked
# ======================================================
class DBCNNBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            ResidualBlock(64, 64),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3),
            ResidualBlock(64, 128, stride=2),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.4),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

class StackedDBCNN(nn.Module):
    def __init__(self, num_branches=3):
        super().__init__()
        self.branches = nn.ModuleList([DBCNNBranch() for _ in range(num_branches)])

    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        return torch.stack(outs, dim=0).mean(dim=0)

# ======================================================
# 6. MobileViTBlock
# ======================================================
class MobileViTBlock(nn.Module):
    def __init__(self, in_channels, out_channels, patch_size=4, transformer_dim=192, num_heads=4, num_layers=2, downsample=False):
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.downsample = downsample

        self.local_rep = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * 4, 1, bias=False),
            nn.BatchNorm2d(in_channels * 4),
            nn.ReLU(),
            nn.Conv2d(in_channels * 4, in_channels * 4, 3, padding=1, groups=in_channels * 4, bias=False),
            nn.BatchNorm2d(in_channels * 4),
            nn.ReLU(),
            nn.Conv2d(in_channels * 4, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )

        self.unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size)
        self.patch_embed = nn.Linear(in_channels * patch_size * patch_size, transformer_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=transformer_dim, nhead=num_heads, dim_feedforward=transformer_dim*2, activation='gelu', batch_first=True, dropout=0.1),
            num_layers=num_layers
        )
        self.patch_proj_back = nn.Linear(transformer_dim, in_channels)

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels * 2, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU()
        )

        self.down = nn.MaxPool2d(2, 2) if downsample else nn.Identity()

    def forward(self, x):
        B, C, H, W = x.shape
        local = self.local_rep(x)

        patches = self.unfold(local)
        patches = patches.transpose(1, 2)
        patches = self.patch_embed(patches)
        patches = self.transformer(patches)
        patches = self.patch_proj_back(patches)
        patches = patches.transpose(1, 2)

        p = self.patch_size
        global_feat = patches.view(B, -1, H // p, W // p)
        global_feat = F.interpolate(global_feat, size=(H, W), mode='nearest')

        fused = torch.cat([local, global_feat], dim=1)
        out = self.fusion(fused)
        out = self.down(out)
        return out

class MobileViT(nn.Module):
    def __init__(self, feature_dim=256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )

        self.stage1 = MobileViTBlock(32, 64, patch_size=4, transformer_dim=128, num_heads=4, downsample=True)
        self.stage2 = MobileViTBlock(64, 128, patch_size=4, transformer_dim=192, num_heads=4, downsample=True)
        self.stage3 = MobileViTBlock(128, 160, patch_size=4, transformer_dim=256, num_heads=8, downsample=False)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(160, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.head(x)
        return x

# ======================================================
# 7. DenseNet121
# ======================================================
class DenseNet121Wrapper(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.densenet121(weights='DEFAULT')
        self.features = base.features
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(1024, 256)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x


# 8.GCBAM
# ======================================================
class GCBAM(nn.Module):
    def __init__(self, feature_dim=256, reduction_ratio=16):
        super().__init__()
        reduced = feature_dim // reduction_ratio

        self.ca = nn.Sequential(
            nn.Linear(feature_dim, reduced),
            nn.ReLU(),
            nn.Linear(reduced, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.Sigmoid()
        )

        self.sa = nn.Sequential(
            nn.Conv1d(1, 1, kernel_size=7, padding=3),
            nn.BatchNorm1d(1),  # Fixed: only 1 channel
            nn.Sigmoid()
        )

        self.gate = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.BatchNorm1d(feature_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.ca(x) * x
        sa_in = ca.unsqueeze(1)  # (B, 1, 256)
        sa = self.sa(sa_in).squeeze(1)  # (B, 256)
        sa_out = ca * sa
        gate = self.gate(sa_out)
        out = x + sa_out * gate
        return out

# ======================================================
# 9. EFFM
# ======================================================
class EFFM(nn.Module):
    def __init__(self, num_branches=3, dim=256):
        super().__init__()
        self.weight_net = nn.Sequential(
            nn.Linear(dim * num_branches, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_branches),
            nn.Softmax(dim=1)
        )

    def forward(self, feats):
        concat = torch.cat(feats, dim=1)
        weights = self.weight_net(concat)
        fused = sum(f * w.unsqueeze(1) for f, w in zip(feats, weights.t()))
        return fused

# ======================================================
# 10. FinalGCBAM
# ======================================================
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.stacked_dbcnn = StackedDBCNN(num_branches=3)
        self.mobilevit     = MobileViT()
        self.densenet      = DenseNet121Wrapper()

        self.hba_dbcnn     = GCBAM()
        self.hba_mobilevit = GCBAM()
        self.hba_densenet  = GCBAM()

        self.fusion = EFFM()
        self.classifier = nn.Linear(256, 1)

    def forward(self, x):
        f1 = self.stacked_dbcnn(x)
        f2 = self.mobilevit(x)
        f3 = self.densenet(x)

        f1 = self.hba_dbcnn(f1)
        f2 = self.hba_mobilevit(f2)
        f3 = self.hba_densenet(f3)

        fused = self.fusion([f1, f2, f3])
        out = self.classifier(fused)
        return out

# ======================================================
# 11. Training Loop (NO MODEL SAVING)
# ======================================================
model = HybridModel().to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

epochs = 50

print("Starting training...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1).float()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss /= len(train_loader)
    train_acc = correct / total

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device).unsqueeze(1).float()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d} | Train L:{train_loss:.4f} A:{train_acc:.4f} | Val L:{val_loss:.4f} A:{val_acc:.4f}")

    torch.cuda.empty_cache()

# ======================================================
# 12. Test Evaluation (using final model weights after training)
# ======================================================
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = (torch.sigmoid(outputs) > 0.5).cpu().numpy().astype(int).flatten()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("\n=== Final Test Results (using model after full training) ===")
print(classification_report(all_labels, all_preds, target_names=['Normal', 'Abnormal']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'])
plt.title('Test Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

gc.collect()
torch.cuda.empty_cache()

print("Training and evaluation complete! (No model weights were saved as requested.)")


Using device: cuda
Original x_data shape: (204, 20, 256, 256, 1)
y_data shape: (204,)
Original x_data shape: (204, 20, 256, 256, 1)
y_data shape: (204,)
Flattened: (4080, 256, 256), labels: (4080,)
Flattened: (4080, 256, 256), labels: (4080,)
Class weights: {0: 1.0, 1: 1.0}
Pos weight: 1.0
Class weights: {0: 1.0, 1: 1.0}
Pos weight: 1.0
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 119MB/s]



Starting training...
Starting training...
Epoch 01 | Train L:0.6218 A:0.6877 | Val L:0.4523 A:0.7941
Epoch 01 | Train L:0.6218 A:0.6877 | Val L:0.4523 A:0.7941
